# CenterPoint (centerpoint branch) -- Colab training

CenterPoint (Yin, Zhou & Krahenbuhl, CVPR 2021) reproduction on the sonar diver
dataset: VoxelNet-style dense VFE/backbone + a single anchor-free CenterHead
(Gaussian-heatmap classification, NMS-free max-pool decode). See this branch's
`README.md` for the full architecture/paper-fidelity notes.

Runtime -> Change runtime type -> GPU (T4 or better), before running anything below.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Clone and install

The model code lives on the `centerpoint` branch specifically.

In [ ]:
import os

# Safe to re-run from any state (fresh /content, already inside the repo, or
# already cloned but not cd'ed in) -- avoids a nested-clone trap
# (3d-point-cloud/3d-point-cloud/...) if this cell runs twice.
if os.path.basename(os.getcwd()) == "3d-point-cloud" and os.path.exists("train.py"):
    print("already inside 3d-point-cloud/ -- nothing to do")
else:
    if not os.path.isdir("3d-point-cloud"):
        !git clone -b centerpoint https://github.com/izione/3d-point-cloud.git
    else:
        print("3d-point-cloud/ already exists here -- skipping clone")
    %cd 3d-point-cloud

!pip install -q -r requirements.txt

Install spconv -- **required** (the paper-faithful backbone is sparse, not dense). Pick the cuXXX tag matching this Colab runtime's CUDA version (check with `!python -c "import torch; print(torch.version.cuda)"` -- cu120/cu121 has worked on recent Colab images; see https://github.com/traveller59/spconv for the full tag list).

In [ ]:
!pip install -q spconv-cu120
import spconv
print("spconv", spconv.__version__)

## 2. Get the dataset

Downloads `dataset.zip` from a Google Drive share link and unzips it onto the
Colab VM's local disk (`/content/dataset_extracted`) -- not the Drive-mounted
path, since the dataset reads many small per-frame files every epoch and local
disk is much faster than Drive's network filesystem for that access pattern.

Paste your `dataset.zip` share link below (Drive -> right-click the file ->
Share -> Copy link; "Anyone with the link" needs to be able to view it).

In [ ]:
DATASET_ZIP_SHARE_URL = "https://drive.google.com/file/d/PASTE_YOUR_FILE_ID_HERE/view?usp=drive_link"

In [ ]:
!pip install -q gdown
!gdown --fuzzy "{DATASET_ZIP_SHARE_URL}" -O /content/dataset.zip
!unzip -q -o /content/dataset.zip -d /content/dataset_extracted
!ls /content/dataset_extracted

Auto-detect the dataset root: `dataset.zip` is expected to contain `Person1/scene_0000/...`
either directly at its top level, or wrapped in one extra folder (e.g. a
`dataset/Person1/...` layout) -- this descends past wrapper folders until it
finds `Person*` folders, then sets `CENTERPOINT_DATA_ROOT` (read by `config.py`
at import time, so this must run before importing anything from this repo).

In [ ]:
import os

DATASET_ROOT = "/content/dataset_extracted"
while True:
    entries = [e for e in os.listdir(DATASET_ROOT) if not e.startswith(".")]
    if any(e.startswith("Person") for e in entries):
        break
    subdirs = [e for e in entries if os.path.isdir(os.path.join(DATASET_ROOT, e))]
    if len(subdirs) != 1:
        raise RuntimeError(
            f"couldn't auto-detect the dataset root under {DATASET_ROOT!r} -- "
            f"found {entries!r}, expected exactly one wrapper folder or Person* folders directly. "
            f"Set DATASET_ROOT by hand and skip this cell's loop."
        )
    DATASET_ROOT = os.path.join(DATASET_ROOT, subdirs[0])
print(f"DATASET_ROOT = {DATASET_ROOT}")

os.environ["CENTERPOINT_DATA_ROOT"] = DATASET_ROOT

No shareable zip link yet, or prefer the manual route? Mount your own Drive
copy instead (slower per-epoch since Drive is a network filesystem, no
zip/link needed):

```python
from google.colab import drive
drive.mount('/content/drive')
os.environ["CENTERPOINT_DATA_ROOT"] = "/content/drive/MyDrive/dataset"  # wherever you uploaded Person*/scene_*/
```

## 3. Mount Drive (for checkpoint/log persistence)

Separate from the dataset above -- this is so checkpoints AND `loss_history.csv`
survive a Colab disconnect (step 6 writes both there).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Sanity check

Runs the full pipeline (dataset -> model -> loss -> backward -> decode -> 3D
IoU) on a handful of real frames from the dataset just fetched above -- catches
path/shape problems in seconds, before committing to a long dataloader/training run.

In [ ]:
!python smoke_test.py

## 5. Quick batch-size/speed check

`config.py`'s `BATCH_SIZE`/`NUM_EPOCHS` were only measured on an RTX 2070 --
Colab GPUs (T4/A100/etc) are different hardware, so time a few real steps here
before committing to a full run.

In [ ]:
import time

import torch
from torch.utils.data import DataLoader

import config
from center_loss import center_voxelnet_loss
from dataset import VoxelDataset, collate_fn, zyaw_flatten
import heatmap_targets as ht
from model import CenterPointVoxelNet
import numpy as np

BATCH_SIZE_TO_TEST = config.BATCH_SIZE   # override here if you want to try a different value
N_WARMUP = 5
N_TIMED = 20

device = torch.device("cuda")
train_ds = VoxelDataset("train")
loader = DataLoader(train_ds, batch_size=BATCH_SIZE_TO_TEST, shuffle=True, collate_fn=collate_fn)
model = CenterPointVoxelNet().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=config.LR, momentum=config.MOMENTUM)
steps_per_epoch = len(loader)
print(f"train frames: {len(train_ds)}  steps/epoch at batch={BATCH_SIZE_TO_TEST}: {steps_per_epoch}")

def run_one_step(it):
    try:
        batch = next(it)
    except StopIteration:
        it = iter(loader)
        batch = next(it)
    pred = model(batch["voxel_features"].to(device), batch["num_points"].to(device),
                 batch["coords"].to(device))
    hm, mask, off, z, dim, rot = [], [], [], [], [], []
    for objects in batch["gt_objects"]:
        t = ht.build_heatmap_targets(zyaw_flatten(objects))
        hm.append(t["heatmap"]); mask.append(t["reg_mask"]); off.append(t["offset"])
        z.append(t["z"]); dim.append(t["dim"]); rot.append(t["rot"])
    target = {"heatmap": torch.from_numpy(np.stack(hm)).to(device),
              "reg_mask": torch.from_numpy(np.stack(mask)).to(device),
              "offset": torch.from_numpy(np.stack(off)).to(device),
              "z": torch.from_numpy(np.stack(z)).to(device),
              "dim": torch.from_numpy(np.stack(dim)).to(device),
              "rot": torch.from_numpy(np.stack(rot)).to(device)}
    loss, _ = center_voxelnet_loss(pred["heatmap"], pred["offset"], pred["z"], pred["dim"], pred["rot"],
                                    target["heatmap"], target["reg_mask"], target["offset"],
                                    target["z"], target["dim"], target["rot"])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return it

it = iter(loader)
torch.cuda.reset_peak_memory_stats()
for _ in range(N_WARMUP):
    it = run_one_step(it)

torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(N_TIMED):
    it = run_one_step(it)
torch.cuda.synchronize()
elapsed = time.perf_counter() - t0

sec_per_step = elapsed / N_TIMED
reserved = torch.cuda.max_memory_reserved() / 1024**3
total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
epoch_min = sec_per_step * steps_per_epoch / 60

print(f"peak_reserved={reserved:.2f} GiB / {total_mem:.1f} GiB ({100*reserved/total_mem:.0f}%)")
print(f"{sec_per_step*1000:.0f} ms/step  ->  ~{epoch_min:.1f} min/epoch  ->  "
      f"~{epoch_min*config.NUM_EPOCHS/60:.1f} hours for all {config.NUM_EPOCHS} epochs "
      f"(config.NUM_EPOCHS -- pass --epochs to train.py to override)")

del model, optimizer, loader, train_ds
torch.cuda.empty_cache()

## 6. Train

Checkpoints AND `loss_history.csv` go to Drive so a disconnect mid-run doesn't
lose progress. Validates on every val-split frame every epoch (cheap here --
CenterPoint's decode is anchor-free/NMS-free).

In [ ]:
DRIVE_CKPT_DIR = "/content/drive/MyDrive/3d-point-cloud-runs/centerpoint"

!python train.py --ckpt_dir "{DRIVE_CKPT_DIR}"
# full-3D rotation target instead of the paper-faithful yaw-only default:
# !python train.py --ckpt_dir "{DRIVE_CKPT_DIR}" --full3d

Resume after a disconnect -- pick the latest `epoch_N.pth` actually present in
`DRIVE_CKPT_DIR` (check with `!ls "{DRIVE_CKPT_DIR}"` if unsure) and continues
training at epoch N+1 for the same total `--epochs`:

In [ ]:
# !ls "{DRIVE_CKPT_DIR}"
# RESUME_FROM = f"{DRIVE_CKPT_DIR}/epoch_7.pth"   # <-- set to the latest checkpoint you have
# !python train.py --ckpt_dir "{DRIVE_CKPT_DIR}" --resume "{RESUME_FROM}"

## 7. Evaluate on the held-out test split

`loss_history.csv` (in `DRIVE_CKPT_DIR`) already has per-epoch train loss and
val AP3D/precision/recall at every IoU threshold -- open it directly
(pandas/plot) for a first look. For the final numbers on the test split with a
chosen checkpoint (best epoch printed at the end of step 6's training log):

In [ ]:
import torch

import config
from dataset import VoxelDataset
from model import CenterPointVoxelNet
from eval import evaluate_full

BEST_CHECKPOINT = f"{DRIVE_CKPT_DIR}/epoch_19.pth"   # <-- set to your best epoch

device = torch.device("cuda")
model = CenterPointVoxelNet().to(device)
ckpt = torch.load(BEST_CHECKPOINT, map_location=device)
model.load_state_dict(ckpt["model"])

test_ds = VoxelDataset("test")
results = evaluate_full(model, device, test_ds)
for thr in config.IOU_THRESHOLDS:
    ap, p, r = results[thr]
    print(f"IoU>={thr}: AP3D={ap:.4f}  P={p:.4f}  R={r:.4f}")